# MegaMart Dirty Data Profiling Report

## 1. Objective

This notebook analyse the quality of all generated MegaMart raw datasets prior to the dbt data cleaning pipeline.

The purpose is to:
- understand the extent of data quality issues
- identify the datasets with the highest level of corruption
- analyse the most common data quality problems
- assess their business impact
- justify the cleaning rules imeplemented later in dbt

All statistics are generated automatically by the dirty data profiling pipeline.

## 2. Imports

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

## 3. Overall Dataset Health

The following table summarises the health of every dirty dataset.

This summary can answer the following business questions:
- Which dataset is the dirtiest?
- Which dataset has the highest health score?
- Which dataset has the most errors?
- Which should be cleaned first?

Datasets with higher dirty ratios require greater cleaning effort.
<br>Health score = 100 - Dirty Ratio

In [2]:
BASE_DIR = Path.cwd()
REPORT_DIR = BASE_DIR / "reports"
summary = pd.read_csv(REPORT_DIR / "summary_report.csv")
summary

,dataset,total_rows,dirty_rows,dirty_ratio,health_score,total_error_types,total_errors,top_error,numeric_columns
0,clickstreams,3214,1183,0.3681,63.19,34,1682,bot traffic,3
1,store_catalogues,235,58,0.2468,75.32,5,64,price anomaly,1
2,customers,50,9,0.1800,82.00,8,12,email formatting anomaly,1
3,transactions,200,34,0.1700,83.00,8,36,calculation error,7
4,product_reviews,208,27,0.1298,87.02,5,28,duplicate row,1
5,products,80,10,0.1250,87.50,5,10,product name formatting anomaly,2
6,competitor_price_history,2025,242,0.1195,88.05,4,254,has_active_promo stored as string,1
7,transaction_items,1748,200,0.1144,88.56,5,210,final item price mismatch,5
8,promotions,80,6,0.0750,92.50,4,6,inverted date range,3
9,stock_snapshots,275,18,0.0655,93.45,4,18,invalid stock status value,0


In [3]:
print(f"Datasets analysed : {len(summary)}")
print(f"Average health score : {summary['health_score'].mean():.2f}")
print(f"Worst dataset : {summary.iloc[0]['dataset']}")
print(f"Best dataset : {summary.iloc[-1]['dataset']}")

Datasets analysed : 22
Average health score : 91.91
Worst dataset : clickstreams
Best dataset : inventory_change_events


# 4. Dataset Deep Dive

In [5]:
def load_report(dataset):
    report = REPORT_DIR / f"{dataset}_report.txt"
    with open(report) as f:
        return f.read()

### Customer Data

The customer dataset contains approximately **18% dirty records**, indicating that nearly one in five customer records contains at least one quality issue.

The most frequently detected issues include:

- Email formatting anomalies
- Missing region values
- Customer name formatting inconsistencies

These issues may reduce the effectiveness of customer segmentation, marketing campaigns and regional sales reporting.

Although the dataset has a moderate health score (82/100), the identified issues should be corrected before loading into the analytics layer.

In [7]:
from IPython.display import Markdown

Markdown(f"""
```text
{load_report("customers_dirty")}
""")


```text
============================================================
DATASET: customers_dirty
============================================================
Total rows: 50
Dirty rows: 9
Dirty ratio: 18.00%
Health score: 82.0/100

TOP ERRORS
email formatting anomaly: 3
missing region: 2
customer name formatting anomaly: 2
email marketing enabled without email: 1
missing email: 1
missing area: 1
missing email_marketing_opt_in: 1
future signup date: 1

ERROR DENSITY
0 errors -> 41 rows
1 errors -> 6 rows
2 errors -> 3 rows

SEVERITY BREAKDOWN
low: 5
unknown: 1
medium: 5
high: 1

VALIDATION CHECKS

NUMERIC PROFILING
loyalty_points: mean=0.0, median=0.0, std=0.0, min=0, max=0

OUTLIERS
loyalty_points: 0 outliers (0.00%)


In [ ]:
customers = pd.read_csv("../dirty_data_generation/dirty_data/customers_dirty.csv")

customers[
    customers.error_types.notna()
]

,customer_id,customer_type,customer_name,email,gender,dob,area,region,signup_date,loyalty_points,customer_segment,email_marketing_opt_in,sms_marketing_opt_in,push_notifications_opt_in,device_category,device_platform,error_types
0,CUST001,Retail Members,NaN,davissheila@example.net,NaN,NaN,NaN,NaN,2023-04-04,0,New Customers,NaN,NaN,NaN,NaN,NaN,['email formatting anomaly']
1,CUST002,Retail Members,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-03,0,High Spenders,True,NaN,NaN,NaN,NaN,['email marketing enabled without email']
2,CUST003,Retail Walk-In,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,[]
3,CUST004,Retail Walk-In,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,[]
4,CUST005,Retail Walk-In,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,[]
5,CUST006,Retail Walk-In,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,[]
6,CUST007,Retail Members,NaN,donald09@example.org,NaN,NaN,NaN,NaN,2025-06-28,0,Active Customers,NaN,NaN,NaN,NaN,NaN,[]
7,CUST008,Retail Walk-In,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,[]
8,CUST009,Online Only,Donna Cruz,DONNA.CRUZ@EXAMPLE.NET,Male,1940-09-15,Lorong Chuan,NaN,2024-03-13,0,Active Customers,False,False,False,Desktop,Web,"['missing region', 'email formatting anomaly']"
9,CUST010,Omnichannel,MICHAEL NGUYEN,michael.nguyen@example.net,Male,1972-05-11,Lakeside,West,2025-02-22,0,Active Customers,False,True,True,Mobile,iOS,['customer name formatting anomaly']


| Detected Issue           | Business Impact                       | dbt Cleaning Rule      |
| ------------------------ | ------------------------------------- | ---------------------- |
| Email formatting anomaly | Marketing emails cannot be delivered  | Regex validation       |
| Missing region           | Regional reporting becomes inaccurate | Default Unknown Region |
| Customer name formatting | Duplicate customer names              | Trim + InitCap         |
| Future signup date       | Customer chronology becomes invalid   | Reject invalid date    |


## 4. Cleaning Strategy

| Dirty Data Issue   | Cleaning Rule        | dbt Implementation |
| ------------------ | -------------------- | ------------------ |
| Leading whitespace | Trim whitespace      | `trim()`           |
| Mixed casing       | Standardise text     | `initcap()`        |
| Empty strings      | Convert to NULL      | `nullif()`         |
| Duplicate rows     | Retain latest record | `row_number()`     |
| Invalid dates      | Safe conversion      | `safe_cast()`      |
| Negative values    | Convert or reject    | `abs()` / filter   |
| Missing values     | Impute or default    | `coalesce()`       |


## 5. Conclusion

The automated profiling process successfully identified data quality issues across all generated datasets.

Key findings include:

- Data quality varies significantly between datasets.
- Several datasets contain high proportions of invalid records.
- Duplicate records, missing values, formatting inconsistencies, and invalid numeric values were common.
- Health scores provide a practical mechanism for prioritising cleaning effort.

The insights generated in this report form the basis for the subsequent implementation of dbt cleaning models and validation tests, ensuring that the final curated datasets are accurate, consistent, and suitable for downstream analytics.